# UFile (and UFileList)

Urgap provids a standardized interface to interact with different file storages and present the file objects in a standardized fashion.

As writing this tutorial the surported file storages are:

- google storage bucket
- azure blob storage
- minio bucket
- samba network drives
- local file paths
- https
- (s)ftp

Since we have abstracted the interaction with the file storages in an interfaces (see. urgap.ufile.io), other storage backends can be added with ease.

Urgap uses uri as pointer to files with a major difference to other systems, that is Urgap separates the location and the identity of a file. Think of it declaring files like books irl, that is each book has a unique ISBN which defines its identity, yet it location, e.g. which bookstore currently holds that book is not unique.

We used the fragment of a URI to specify the identity of a file, and the last element of the netloc as container, e.g.

```bash
    uri: file://<any directory structure ...>/<container>#<object>
                                                           ^---- (fragment)
                                                    ^-(url, last element is container name)
```

In [1]:
import urgap

2024-09-13 12:47:58,577 - __init__.py - 83 - INFO - 
              .__________________________________________________________.
               \     |   \    __    \      ___/      ____/    _    |     |_____. 
                |    |    |   |/    /\        \          |    _    |     |  -cf|
                |_________|___|\    \_________/\____|    |____|____|___________|
                                \_____/             |____|
        
                uncle      reels     shrilly     gifted     across     laugh  
    
  version                : 2.5.17.dev0
  ursgal home            : /Users/cf322940/.ursgal
  project_folder         : /Users/cf322940/dev/ursgal2/docs/source/tutorials
  ursgal config          : /Users/cf322940/.ursgal/ursgal.json
  scratch disk           : /Users/cf322940/tmp/u_skinny_trick_presignifies_alarmed_bed



Within the urgap home diretory, there is a file called credentials_lookup.json

In [5]:
import json
from pathlib import Path
uc = json.load(
    open(
        Path(urgap.home / "credentials_lookup.json")
    )
)

In [8]:
uc

{'description': 'Autogenerated Ursgal Credential lookup',
 'credentials': [{'description': 'OMIQ Api',
   'scheme': 'omiq',
   'host': 'gsk',
   'user': 'U_OMIQ_USER',
   'password': 'U_OMIQ_PASSWORD',
   'secure': True,
   'secret_store': 'env',
   'cloud_host_pid': 'gsk.omiq.ai'},
  {'description': 'gcs using libcloud does not need host yet schema+host is used for internal lookups',
   'scheme': 'gcs-libcloud',
   'host': 'gsk-rd-ngs-sbx',
   'user': 'U_GCS_USER',
   'password': 'U_GCS_PASSWORD',
   'secure': True,
   'secret_store': 'env',
   'cloud_host_pid': 'gsk-rd-ngs-sbx'},
  {'description': 'gcs using libcloud does not need host yet schema+host is used for internal lookups',
   'scheme': 'gcs-libcloud',
   'host': 'gsk-tech-dso-j2c-eng',
   'user': 'U_GCS_USER_2',
   'password': 'U_GCS_PASSWORD_2',
   'secure': True,
   'secret_store': 'env',
   'cloud_host_pid': 'gsk-tech-dso-j2c-eng'},
  {'description': 'local minio server',
   'scheme': 'minio',
   'host': 'localhost:9000',

This file is used to point the urgap credential manager to the right secret store to extract the credentials. Take for example this entry in the `uc["credentials"]`
```
{
    'description': 'gcs using libcloud does not need host yet schema+host is used for internal lookups',
    'scheme': 'gcs-libcloud',
    'host': 'gsk-rd-ngs-sbx',
    'user': 'U_GCS_USER',
    'password': 'U_GCS_PASSWORD',
    'secure': True,
    'secret_store': 'env',
    'cloud_host_pid': 'gsk-rd-ngs-sbx'
}
```

if a uri or connection string has the schema `gcs-libcloud` and points the the host `gsk-rd-ngs-sbx`, then the secret manager will look into the secret_store `env` and extract the user/login from the variable under `U_GCS_USER` the password from the variable under `U_GCS_PASSWORD`.

urgap will initialze a credential manager under
`urgap.instances.ucredential_manager` during init.

We can extract the credentials using the the methods `.extract_credentials`, `.get_password` or `.get_user`.

We can also supply more credentials dynamically using the `.add_credentials` methods. For example:

In [9]:
um = urgap.UCredentialManager()
um.add_credentials(
    [
        {
            'description': 'Demo1',
            'scheme': 'dog',
            'host': 'town',
            'user': 'U_DOG_USER',
            'password': 'U_DOG_PASSWORD',
            'secure': True,
            'secret_store': 'env',
            'cloud_host_pid': 'gsk-rd-ngs-sbx'
        }
    ]
)

Let set those env variables now.

In [11]:
import os
os.environ["U_DOG_USER"] = "d0g-name"
os.environ["U_DOG_PASSWORD"] = "d0g-password"

In [12]:
um.get_password("dog://town")

'd0g-password'

In [13]:
um.extract_credentials("dog://town")

{'user': 'd0g-name', 'password': 'd0g-password'}